# Set models and coverage-dependent magnitude estimation

This notebook reproduces the principal MeanPool-MLP, Deep Sets, and Set Transformer experiments and the coverage-stratified analyses reported in the associated manuscript.

Set the `SEISMIC_DATA_DIR` environment variable to the directory containing `features_socal_full.csv` and `full_stations.csv`. Outputs are written to `SEISMIC_OUTPUT_DIR` when defined, otherwise to `../results/generated/set_models`.


In [ ]:
# %% Cell 1 — Imports and reproducibility
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:
# %% Cell 2 — Configuration + load data
import os

DATA_DIR = Path(os.environ.get("SEISMIC_DATA_DIR", "../data/processed"))
OUT_DIR = Path(os.environ.get("SEISMIC_OUTPUT_DIR", "../results/generated/set_models"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25
LR = 1e-3
WEIGHT_DECAY = 1e-4

CHANNELS = ['E', 'N', 'Z']
KEEP_FEATURES = [
    'rms', 'peak', 'crest_factor',
    'spectral_bandwidth', 'spectral_centroid', 'spectral_entropy',
    'zcr', 'energy_ratio_early_late',
    'band_5_10Hz', 'band_20_50Hz'
]

df = pd.read_csv(DATA_DIR / "features_socal_full.csv")
stations = pd.read_csv(DATA_DIR / "full_stations.csv")

station_names = stations['receiver_code'].astype(str).values
station_to_idx = {name: i for i, name in enumerate(station_names)}

feat_cols = [f'{feat}_{ch}' for ch in CHANNELS for feat in KEEP_FEATURES]
feat_cols = [c for c in feat_cols if c in df.columns]
N_FEATURES = len(feat_cols)

df['receiver_code'] = df['receiver_code'].astype(str)
df = df[df['receiver_code'].isin(set(station_names))].copy()

ev_counts = df.groupby('source_id')['receiver_code'].nunique()
df = df[df['source_id'].isin(ev_counts[ev_counts >= 2].index)].copy()

for col in feat_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

missing = df[feat_cols].isna().sum().sum()
if missing != 0:
    raise ValueError(
        f"Expected complete final feature table, but found {missing} missing values. "
        "Rebuild the processed feature table before running the final experiment."
    )

print(f"Stations in infrastructure: {len(station_names)}")
print(f"Eligible events: {df['source_id'].nunique():,}")
print(f"Traces: {len(df):,}")
print(f"Waveform features per station: {N_FEATURES}")
print("Missing feature values: 0")


## Event representation

Each earthquake is an unordered set
\[
\mathcal{S}_e=\{\mathbf{x}_{e,1},\ldots,\mathbf{x}_{e,n_e}\}.
\]

No edges are supplied.

- **MeanPool-MLP:** mean-pools standardized station features, then predicts magnitude.
- **Deep Sets:** learns a nonlinear station encoder before pooling.
- **Set Transformer:** learns self-attention among the active stations before pooling.


In [ ]:
# %% Cell 3 — Build event-level raw sets
event_records = []

for event_id, g in df.groupby('source_id', sort=False):
    agg_map = {c: 'mean' for c in feat_cols}
    agg_map['source_magnitude'] = 'first'
    g2 = g.groupby('receiver_code', as_index=False).agg(agg_map)

    X = g2[feat_cols].to_numpy(dtype=np.float32)
    station_codes = g2['receiver_code'].astype(str).to_numpy()
    y = np.float32(g2['source_magnitude'].iloc[0])

    if len(X) >= 2:
        event_records.append({
            'event_id': event_id,
            'X': X,
            'station_codes': station_codes,
            'y': y,
            'n_stations': len(X),
        })

event_ids = np.array([r['event_id'] for r in event_records], dtype=object)
coverage = np.array([r['n_stations'] for r in event_records], dtype=int)
targets = np.array([r['y'] for r in event_records], dtype=np.float32)

print(f"Built {len(event_records):,} event sets")
print(pd.Series(coverage).describe(percentiles=[.5, .75, .9, .95, .99]).to_string())
print("\nCoverage counts:")
print(pd.Series(coverage).value_counts().sort_index().head(20).to_string())


In [ ]:
# %% Cell 4 — Dataset + padding collate
class EventSetDataset(Dataset):
    def __init__(self, records, scaler=None):
        self.records = records
        self.scaler = scaler

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        X = r['X']
        if self.scaler is not None:
            X = self.scaler.transform(X).astype(np.float32)
        return {
            'x': torch.tensor(X, dtype=torch.float32),
            'y': torch.tensor(r['y'], dtype=torch.float32),
            'event_id': r['event_id'],
            'n_stations': r['n_stations'],
        }

def collate_sets(batch):
    B = len(batch)
    lengths = torch.tensor([b['x'].shape[0] for b in batch], dtype=torch.long)
    max_n = int(lengths.max().item())
    feat_dim = batch[0]['x'].shape[1]

    x = torch.zeros(B, max_n, feat_dim, dtype=torch.float32)
    mask = torch.zeros(B, max_n, dtype=torch.bool)
    y = torch.stack([b['y'] for b in batch])

    event_id = []
    n_stations = []
    for i, b in enumerate(batch):
        n = b['x'].shape[0]
        x[i, :n] = b['x']
        mask[i, :n] = True
        event_id.append(b['event_id'])
        n_stations.append(b['n_stations'])

    return {
        'x': x,
        'mask': mask,
        'y': y,
        'event_id': event_id,
        'n_stations': torch.tensor(n_stations, dtype=torch.long),
    }

def masked_mean(x, mask):
    w = mask.unsqueeze(-1).to(x.dtype)
    return (x * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)


In [ ]:
import sys
import platform

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import torch

print("=" * 60)
print("REPRODUCIBILITY ENVIRONMENT")
print("=" * 60)

print(f"Python:         {sys.version}")
print(f"Platform:       {platform.platform()}")

print("\nCore packages")
print(f"NumPy:          {np.__version__}")
print(f"Pandas:         {pd.__version__}")
print(f"SciPy:          {scipy.__version__}")
print(f"scikit-learn:   {sklearn.__version__}")
print(f"Matplotlib:     {matplotlib.__version__}")

print("\nPyTorch")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA runtime:   {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")

try:
    import torch_geometric
    print(f"PyG:            {torch_geometric.__version__}")
except ImportError:
    print("PyG:            not installed")

try:
    import networkx as nx
    print(f"NetworkX:       {nx.__version__}")
except ImportError:
    print("NetworkX:       not installed")

print("=" * 60)

In [ ]:
# %% Cell 5 — Models
class MeanPoolMLP(nn.Module):
    def __init__(self, in_dim, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, mask):
        pooled = masked_mean(x, mask)
        return self.net(pooled).squeeze(-1)


class DeepSets(nn.Module):
    def __init__(self, in_dim, latent=128, dropout=0.2):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, latent),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(latent, latent),
            nn.ReLU(),
        )
        self.rho = nn.Sequential(
            nn.Linear(latent, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x, mask):
        h = self.phi(x)
        pooled = masked_mean(h, mask)
        return self.rho(pooled).squeeze(-1)


class SAB(nn.Module):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 2 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * dim, dim),
        )
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        key_padding_mask = ~mask
        a, _ = self.attn(
            x, x, x,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )
        x = self.norm1(x + self.dropout(a))
        f = self.ff(x)
        x = self.norm2(x + self.dropout(f))
        x = x * mask.unsqueeze(-1).to(x.dtype)
        return x


class SetTransformerRegressor(nn.Module):
    def __init__(self, in_dim, dim=128, heads=4, n_blocks=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            SAB(dim, heads=heads, dropout=dropout)
            for _ in range(n_blocks)
        ])
        self.head = nn.Sequential(
            nn.Linear(dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, x, mask):
        h = self.input_proj(x)
        h = h * mask.unsqueeze(-1).to(h.dtype)
        for block in self.blocks:
            h = block(h, mask)
        pooled = masked_mean(h, mask)
        return self.head(pooled).squeeze(-1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, model in {
    'MeanPool-MLP': MeanPoolMLP(N_FEATURES),
    'DeepSets': DeepSets(N_FEATURES),
    'SetTransformer': SetTransformerRegressor(N_FEATURES),
}.items():
    print(f"{name:16s}: {count_parameters(model):,} trainable parameters")


In [ ]:
# %% Cell 6 — Fold-safe scaler + training/evaluation
def fit_scaler_from_records(records):
    X_train = np.concatenate([r['X'] for r in records], axis=0)
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def predict_loader(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            y = batch['y'].to(DEVICE)

            pred = model(x, mask)

            for eid, yt, yp, nst in zip(
                batch['event_id'],
                y.cpu().numpy(),
                pred.cpu().numpy(),
                batch['n_stations'].numpy()
            ):
                rows.append({
                    'event_id': eid,
                    'y_true': float(yt),
                    'y_pred': float(yp),
                    'n_stations': int(nst),
                })
    return pd.DataFrame(rows)


def train_model(model, train_records, val_records,
                lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE,
                patience=PATIENCE):

    scaler = fit_scaler_from_records(train_records)
    train_ds = EventSetDataset(train_records, scaler=scaler)
    val_ds = EventSetDataset(val_records, scaler=scaler)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_sets, num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sets, num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = ReduceLROnPlateau(
        optimizer, mode='min', patience=8, factor=0.5
    )
    criterion = nn.MSELoss()

    best_val = np.inf
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for batch in train_loader:
            x = batch['x'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            y = batch['y'].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x, mask)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss_sum += loss.item() * len(y)
            train_n += len(y)

        model.eval()
        val_loss_sum = 0.0
        val_n = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                y = batch['y'].to(DEVICE)
                pred = model(x, mask)
                loss = criterion(pred, y)
                val_loss_sum += loss.item() * len(y)
                val_n += len(y)

        train_loss = train_loss_sum / max(train_n, 1)
        val_loss = val_loss_sum / max(val_n, 1)
        scheduler.step(val_loss)
        history.append((epoch, train_loss, val_loss))

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    pred_df = predict_loader(model, val_loader)

    return pred_df, pd.DataFrame(
        history, columns=['epoch', 'train_mse', 'val_mse']
    ), epoch


def regression_metrics(d):
    y = d['y_true'].to_numpy()
    p = d['y_pred'].to_numpy()
    return {
        'N': len(d),
        'MAE': mean_absolute_error(y, p),
        'RMSE': np.sqrt(mean_squared_error(y, p)),
        'R2': r2_score(y, p) if len(d) >= 2 else np.nan,
    }


In [ ]:
# %% Cell 7 — Run 5-fold experiments
gkf = GroupKFold(n_splits=N_FOLDS)
split_indices = list(
    gkf.split(
        X=np.zeros(len(event_records)),
        y=targets,
        groups=event_ids
    )
)

MODEL_FACTORIES = {
    'MeanPool-MLP': lambda: MeanPoolMLP(N_FEATURES, hidden=128, dropout=0.2),
    'DeepSets': lambda: DeepSets(N_FEATURES, latent=128, dropout=0.2),
    'SetTransformer': lambda: SetTransformerRegressor(
        N_FEATURES, dim=128, heads=4, n_blocks=2, dropout=0.1
    ),
}

all_results = {}
fold_summary = []

for model_name, factory in MODEL_FACTORIES.items():
    print("\n" + "=" * 72)
    print(model_name)
    print("=" * 72)

    model_preds = []

    for fold, (train_idx, val_idx) in enumerate(split_indices):
        fold_seed = SEED + fold
        random.seed(fold_seed)
        np.random.seed(fold_seed)
        torch.manual_seed(fold_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(fold_seed)

        train_records = [event_records[i] for i in train_idx]
        val_records = [event_records[i] for i in val_idx]

        model = factory()
        pred_df, hist_df, n_epochs = train_model(
            model, train_records, val_records
        )
        pred_df['fold'] = fold
        pred_df['model'] = model_name
        model_preds.append(pred_df)

        met = regression_metrics(pred_df)
        fold_summary.append({
            'Model': model_name,
            'Fold': fold,
            **met,
            'Epochs': n_epochs,
        })

        print(
            f"Fold {fold}: "
            f"MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, "
            f"R²={met['R2']:.4f}, epochs={n_epochs}"
        )

    all_results[model_name] = pd.concat(model_preds, ignore_index=True)

predictions = pd.concat(all_results.values(), ignore_index=True)
fold_results = pd.DataFrame(fold_summary)

predictions.to_csv(OUT_DIR / "predictions_all_models.csv", index=False)
fold_results.to_csv(OUT_DIR / "fold_metrics.csv", index=False)

print("\nSaved:")
print(" ", OUT_DIR / "predictions_all_models.csv")
print(" ", OUT_DIR / "fold_metrics.csv")


In [ ]:
# %% Cell 8 — Overall comparison
overall_rows = []
for model_name, d in all_results.items():
    met = regression_metrics(d)
    overall_rows.append({'Model': model_name, **met})

overall = pd.DataFrame(overall_rows).sort_values('MAE').reset_index(drop=True)

print("\nE4 set-model results — directly comparable:")
print(overall.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

overall.to_csv(OUT_DIR / "overall_metrics.csv", index=False)


In [ ]:
# %% Cell 9 — Coverage-stratified evaluation
def coverage_bin(n):
    if n == 2:
        return '2'
    if n == 3:
        return '3'
    if n == 4:
        return '4'
    if n == 5:
        return '5'
    return '>=6'

coverage_rows = []

for model_name, d0 in all_results.items():
    d = d0.copy()
    d['Coverage'] = d['n_stations'].map(coverage_bin)

    for cov in ['2', '3', '4', '5', '>=6']:
        dc = d[d['Coverage'] == cov]
        if len(dc) == 0:
            continue
        met = regression_metrics(dc)
        coverage_rows.append({
            'Model': model_name,
            'Coverage': cov,
            **met
        })

coverage_metrics = pd.DataFrame(coverage_rows)
coverage_metrics['Coverage'] = pd.Categorical(
    coverage_metrics['Coverage'],
    categories=['2', '3', '4', '5', '>=6'],
    ordered=True
)
coverage_metrics = coverage_metrics.sort_values(['Coverage', 'MAE'])

print(coverage_metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
coverage_metrics.to_csv(OUT_DIR / "coverage_stratified_metrics.csv", index=False)


In [ ]:
# %% Cell 10 — Paired bootstrap: model differences in absolute error
def paired_bootstrap_delta_mae(pred_a, pred_b, n_boot=5000, seed=SEED):
    a = pred_a[['event_id', 'y_true', 'y_pred']].rename(columns={'y_pred':'pred_a'})
    b = pred_b[['event_id', 'y_true', 'y_pred']].rename(columns={'y_pred':'pred_b'})
    m = a.merge(b[['event_id', 'pred_b']], on='event_id', how='inner')

    ea = np.abs(m['y_true'].to_numpy() - m['pred_a'].to_numpy())
    eb = np.abs(m['y_true'].to_numpy() - m['pred_b'].to_numpy())
    diff = ea - eb

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    n = len(diff)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        boot[k] = diff[idx].mean()

    return {
        'N': n,
        'Delta_MAE': diff.mean(),
        'CI95_low': np.quantile(boot, 0.025),
        'CI95_high': np.quantile(boot, 0.975),
        'P_boot_A_better': np.mean(boot < 0),
    }

pairs = [
    ('DeepSets', 'MeanPool-MLP'),
    ('SetTransformer', 'MeanPool-MLP'),
    ('SetTransformer', 'DeepSets'),
]

boot_rows = []
for a, b in pairs:
    out = paired_bootstrap_delta_mae(all_results[a], all_results[b])
    boot_rows.append({'A': a, 'B': b, **out})

bootstrap_results = pd.DataFrame(boot_rows)
print(bootstrap_results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
bootstrap_results.to_csv(OUT_DIR / "paired_bootstrap_delta_mae.csv", index=False)


In [ ]:
# %% Cell 11 — Does model advantage change with coverage?
coverage_pair_rows = []

for pair_idx, (a, b) in enumerate(pairs):
    da = all_results[a].copy()
    db = all_results[b].copy()
    da['Coverage'] = da['n_stations'].map(coverage_bin)
    db['Coverage'] = db['n_stations'].map(coverage_bin)

    for cov_idx, cov in enumerate(['2', '3', '4', '5', '>=6']):
        xa = da[da['Coverage'] == cov]
        xb = db[db['Coverage'] == cov]
        if len(xa) < 30 or len(xb) < 30:
            continue

        out = paired_bootstrap_delta_mae(
            xa, xb, n_boot=3000,
            seed=SEED + 100 * pair_idx + cov_idx
        )
        coverage_pair_rows.append({
            'A': a, 'B': b, 'Coverage': cov, **out
        })

coverage_bootstrap = pd.DataFrame(coverage_pair_rows)
if len(coverage_bootstrap):
    coverage_bootstrap['Coverage'] = pd.Categorical(
        coverage_bootstrap['Coverage'],
        categories=['2', '3', '4', '5', '>=6'],
        ordered=True
    )
    coverage_bootstrap = coverage_bootstrap.sort_values(['A', 'B', 'Coverage'])
    print(coverage_bootstrap.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
    coverage_bootstrap.to_csv(
        OUT_DIR / "coverage_paired_bootstrap.csv", index=False
    )


## Interpretation rules

### Outcome 1 — Deep Sets wins, Set Transformer does not
Nonlinear station encoding before pooling matters, but explicit station-to-station interactions are unnecessary at the observed coverage.

### Outcome 2 — Set Transformer improves mainly at higher coverage
This directly supports a coverage-threshold interpretation: relational modeling becomes useful only after enough stations observe the same event.

### Outcome 3 — MeanPool-MLP remains best
Then the negative GNN result is not specifically caused by graph topology. The per-station features already contain most of the predictive signal and relational modeling adds little under sparse sampling.

The coverage-stratified curves are therefore more informative than a single global ranking.


Cell 13 — Fine-grained coverage analysis

In [ ]:
# %% Cell 13 — Fine-grained coverage analysis
import numpy as np
import pandas as pd

PRED_PATH = OUT_DIR / "predictions_all_models.csv"

pred = pd.read_csv(PRED_PATH)

print("Columns:")
print(pred.columns.tolist())
print("\nTotal rows:", len(pred))

# ------------------------------------------------------------
# Padronizar nomes
# ------------------------------------------------------------
pred = pred.rename(columns={
    'model': 'Model'
})

required = [
    'event_id',
    'Model',
    'y_true',
    'y_pred',
    'n_stations',
    'fold'
]

missing = [c for c in required if c not in pred.columns]

if missing:
    raise RuntimeError(
        f"Missing columns: {missing}\n"
        f"Available: {pred.columns.tolist()}"
    )

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
print("\nModels:")
print(pred['Model'].value_counts())

print("\nUnique events:", pred['event_id'].nunique())

print("\nStation-count distribution:")
print(
    pred.drop_duplicates('event_id')['n_stations']
        .value_counts()
        .sort_index()
)

# ------------------------------------------------------------
# Errors
# ------------------------------------------------------------
pred['abs_error'] = np.abs(
    pred['y_true'] - pred['y_pred']
)

pred['sq_error'] = (
    pred['y_true'] - pred['y_pred']
)**2

# ------------------------------------------------------------
# Fine coverage bins
# ------------------------------------------------------------
def coverage_bin(n):
    n = int(n)

    if n <= 10:
        return str(n)
    elif n <= 15:
        return "11-15"
    else:
        return ">15"

pred['coverage_fine'] = (
    pred['n_stations']
    .apply(coverage_bin)
)

coverage_order = (
    [str(i) for i in range(2, 11)]
    + ['11-15', '>15']
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
rows = []

for model in pred['Model'].unique():

    dm = pred[pred['Model'] == model]

    for cov in coverage_order:

        d = dm[
            dm['coverage_fine'] == cov
        ]

        if len(d) == 0:
            continue

        mae = d['abs_error'].mean()

        rmse = np.sqrt(
            d['sq_error'].mean()
        )

        ss_res = np.sum(
            (d['y_true'] - d['y_pred'])**2
        )

        ss_tot = np.sum(
            (d['y_true'] - d['y_true'].mean())**2
        )

        r2 = (
            1 - ss_res / ss_tot
            if ss_tot > 0
            else np.nan
        )

        rows.append({
            'Model': model,
            'Coverage': cov,
            'N': len(d),
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

fine_cov = pd.DataFrame(rows)

fine_cov['Coverage'] = pd.Categorical(
    fine_cov['Coverage'],
    categories=coverage_order,
    ordered=True
)

fine_cov = fine_cov.sort_values(
    ['Coverage', 'Model']
).reset_index(drop=True)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
print("\n" + "="*80)
print("FINE-GRAINED COVERAGE RESULTS")
print("="*80)

print(
    fine_cov.to_string(
        index=False,
        formatters={
            'MAE': '{:.5f}'.format,
            'RMSE': '{:.5f}'.format,
            'R2': '{:.4f}'.format
        }
    )
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
save_path = (
    OUT_DIR /
    "fine_coverage_metrics.csv"
)

fine_cov.to_csv(
    save_path,
    index=False
)

print("\nSaved:", save_path)

Cell 14 — Paired bootstrap: SetTransformer vs DeepSets by coverage

In [ ]:
# %% Cell 14 — Paired bootstrap: SetTransformer vs DeepSets by coverage
import numpy as np
import pandas as pd

N_BOOT = 3000
RNG = np.random.default_rng(42)

# Identificar coluna de ID do evento
candidate_id_cols = [
    'event_id',
    'EventID',
    'source_id',
    'trace_name',
    'sample_id',
    'index'
]

event_col = None

for c in candidate_id_cols:
    if c in pred.columns:
        event_col = c
        break

if event_col is None:
    raise RuntimeError(
        "Não encontrei uma coluna de ID de evento em predictions_all_models.csv.\n"
        f"Colunas disponíveis: {pred.columns.tolist()}"
    )

print("Event ID column:", event_col)

# Separar os dois modelos
ds = pred[pred['Model'] == 'DeepSets'].copy()
st = pred[pred['Model'] == 'SetTransformer'].copy()

# Renomear erros
ds = ds.rename(columns={'abs_error': 'AE_DS'})
st = st.rename(columns={'abs_error': 'AE_ST'})

# Pareamento pelo evento
paired = ds[
    [event_col, 'coverage_fine', 'n_stations', 'AE_DS']
].merge(
    st[
        [event_col, 'AE_ST']
    ],
    on=event_col,
    how='inner'
)

paired['delta'] = paired['AE_ST'] - paired['AE_DS']

print("\nPaired events:", len(paired))

coverage_order = [str(i) for i in range(2, 11)] + ['11-15', '>15']

boot_rows = []

for cov in coverage_order:

    d = paired[paired['coverage_fine'] == cov].copy()

    if len(d) < 30:
        print(f"Skipping coverage {cov}: N={len(d)}")
        continue

    delta = d['delta'].to_numpy()
    n = len(delta)

    observed = delta.mean()

    boot = np.empty(N_BOOT)

    for b in range(N_BOOT):
        idx = RNG.integers(0, n, n)
        boot[b] = delta[idx].mean()

    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    p_st_better = np.mean(boot < 0)

    boot_rows.append({
        'Coverage': cov,
        'N': n,
        'Delta_MAE_ST_minus_DS': observed,
        'CI95_low': ci_low,
        'CI95_high': ci_high,
        'P_boot_ST_better': p_st_better
    })

boot_fine = pd.DataFrame(boot_rows)

boot_fine['Coverage'] = pd.Categorical(
    boot_fine['Coverage'],
    categories=coverage_order,
    ordered=True
)

boot_fine = boot_fine.sort_values('Coverage')

print("\nSetTransformer − DeepSets by coverage:")
print(
    boot_fine.to_string(
        index=False,
        formatters={
            'Delta_MAE_ST_minus_DS': '{:+.5f}'.format,
            'CI95_low': '{:+.5f}'.format,
            'CI95_high': '{:+.5f}'.format,
            'P_boot_ST_better': '{:.4f}'.format
        }
    )
)

boot_fine.to_csv(
    OUT_DIR / "bootstrap_ST_vs_DS_fine_coverage.csv",
    index=False
)

Cell 15 — Coverage-dependent transition figure

Cell 16 — Sparse / Intermediate / Dense regime analysis

In [ ]:
# %% Cell 16 — Sparse / Intermediate / Dense regime analysis
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define coverage regimes
# ------------------------------------------------------------
def coverage_regime(n):
    n = int(n)

    if 2 <= n <= 5:
        return "Sparse (2-5)"
    elif 6 <= n <= 10:
        return "Intermediate (6-10)"
    else:
        return "Dense (>=11)"


pred['coverage_regime'] = (
    pred['n_stations']
    .apply(coverage_regime)
)

regime_order = [
    "Sparse (2-5)",
    "Intermediate (6-10)",
    "Dense (>=11)"
]

# ------------------------------------------------------------
# 2. Métricas por modelo e regime
# ------------------------------------------------------------
rows = []

for model in ['MeanPool-MLP', 'DeepSets', 'SetTransformer']:

    dm = pred[pred['Model'] == model].copy()

    for regime in regime_order:

        d = dm[
            dm['coverage_regime'] == regime
        ]

        if len(d) == 0:
            continue

        mae = np.mean(
            np.abs(d['y_true'] - d['y_pred'])
        )

        rmse = np.sqrt(
            np.mean(
                (d['y_true'] - d['y_pred'])**2
            )
        )

        ss_res = np.sum(
            (d['y_true'] - d['y_pred'])**2
        )

        ss_tot = np.sum(
            (d['y_true'] - d['y_true'].mean())**2
        )

        r2 = (
            1 - ss_res / ss_tot
            if ss_tot > 0
            else np.nan
        )

        rows.append({
            'Model': model,
            'Regime': regime,
            'N': len(d),
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })


regime_metrics = pd.DataFrame(rows)

regime_metrics['Regime'] = pd.Categorical(
    regime_metrics['Regime'],
    categories=regime_order,
    ordered=True
)

regime_metrics = (
    regime_metrics
    .sort_values(['Regime', 'Model'])
    .reset_index(drop=True)
)

print("\n" + "="*90)
print("COVERAGE-REGIME METRICS")
print("="*90)

print(
    regime_metrics.to_string(
        index=False,
        formatters={
            'MAE': '{:.5f}'.format,
            'RMSE': '{:.5f}'.format,
            'R2': '{:.4f}'.format
        }
    )
)

# ------------------------------------------------------------
# 3. Parear DeepSets e SetTransformer por evento
# ------------------------------------------------------------
ds = pred[
    pred['Model'] == 'DeepSets'
][
    ['event_id',
     'coverage_regime',
     'n_stations',
     'y_true',
     'y_pred']
].copy()

st = pred[
    pred['Model'] == 'SetTransformer'
][
    ['event_id',
     'y_pred']
].copy()

ds = ds.rename(
    columns={'y_pred': 'y_pred_DS'}
)

st = st.rename(
    columns={'y_pred': 'y_pred_ST'}
)

paired = ds.merge(
    st,
    on='event_id',
    how='inner'
)

paired['AE_DS'] = np.abs(
    paired['y_true']
    - paired['y_pred_DS']
)

paired['AE_ST'] = np.abs(
    paired['y_true']
    - paired['y_pred_ST']
)

paired['delta'] = (
    paired['AE_ST']
    - paired['AE_DS']
)

# delta < 0 -> SetTransformer better
# delta > 0 -> DeepSets better

# ------------------------------------------------------------
# 4. Paired bootstrap por regime
# ------------------------------------------------------------
N_BOOT = 5000
RNG = np.random.default_rng(42)

boot_rows = []

for regime in regime_order:

    d = paired[
        paired['coverage_regime'] == regime
    ].copy()

    delta = d['delta'].to_numpy()

    ae_ds = d['AE_DS'].to_numpy()
    ae_st = d['AE_ST'].to_numpy()

    n = len(d)

    mae_ds = ae_ds.mean()
    mae_st = ae_st.mean()

    observed_delta = delta.mean()

    # melhoria relativa do ST sobre DS
    # positiva => ST reduz MAE
    relative_gain_pct = (
        (mae_ds - mae_st)
        / mae_ds
        * 100
    )

    boot_delta = np.empty(N_BOOT)

    for b in range(N_BOOT):

        idx = RNG.integers(
            0,
            n,
            size=n
        )

        boot_delta[b] = (
            delta[idx].mean()
        )

    ci_low, ci_high = np.percentile(
        boot_delta,
        [2.5, 97.5]
    )

    p_st_better = np.mean(
        boot_delta < 0
    )

    boot_rows.append({
        'Regime': regime,
        'N': n,
        'MAE_DeepSets': mae_ds,
        'MAE_SetTransformer': mae_st,
        'Delta_MAE_ST_minus_DS': observed_delta,
        'Relative_gain_ST_pct': relative_gain_pct,
        'CI95_low': ci_low,
        'CI95_high': ci_high,
        'P_boot_ST_better': p_st_better
    })


regime_bootstrap = pd.DataFrame(
    boot_rows
)

regime_bootstrap['Regime'] = pd.Categorical(
    regime_bootstrap['Regime'],
    categories=regime_order,
    ordered=True
)

regime_bootstrap = (
    regime_bootstrap
    .sort_values('Regime')
    .reset_index(drop=True)
)

print("\n" + "="*90)
print("DEEPSETS vs SETTRANSFORMER — COVERAGE REGIMES")
print("="*90)

print(
    regime_bootstrap.to_string(
        index=False,
        formatters={
            'MAE_DeepSets':
                '{:.5f}'.format,

            'MAE_SetTransformer':
                '{:.5f}'.format,

            'Delta_MAE_ST_minus_DS':
                '{:+.5f}'.format,

            'Relative_gain_ST_pct':
                '{:+.2f}'.format,

            'CI95_low':
                '{:+.5f}'.format,

            'CI95_high':
                '{:+.5f}'.format,

            'P_boot_ST_better':
                '{:.4f}'.format
        }
    )
)

# ------------------------------------------------------------
# 5. Fração do dataset em cada regime
# ------------------------------------------------------------
event_regime = (
    pred[
        pred['Model'] == 'DeepSets'
    ][
        ['event_id', 'coverage_regime']
    ]
    .drop_duplicates('event_id')
)

regime_counts = (
    event_regime['coverage_regime']
    .value_counts()
    .reindex(regime_order)
)

regime_fraction = (
    regime_counts
    / regime_counts.sum()
)

print("\n" + "="*90)
print("DATASET COMPOSITION")
print("="*90)

for regime in regime_order:

    n = int(regime_counts.loc[regime])

    pct = (
        100
        * regime_fraction.loc[regime]
    )

    print(
        f"{regime:<22}"
        f"N={n:>6d}   "
        f"{pct:>6.2f}%"
    )

# ------------------------------------------------------------
# 6. Salvar tabelas finais
# ------------------------------------------------------------
path_metrics = (
    OUT_DIR /
    "coverage_regime_metrics.csv"
)

path_boot = (
    OUT_DIR /
    "coverage_regime_DS_vs_ST.csv"
)

regime_metrics.to_csv(
    path_metrics,
    index=False
)

regime_bootstrap.to_csv(
    path_boot,
    index=False
)

print("\nSaved:")
print(" ", path_metrics)
print(" ", path_boot)

Cell 17 — Robustness of coverage-regime boundaries

In [ ]:
# %% Cell 17 — Robustness of coverage-regime boundaries
"""
Robustness check for the coverage-dependent relational effect.

Tests:
A) Main:        Sparse 2-5 | Intermediate 6-10 | Dense >=11
B) Alternative: Sparse 2-4 | Intermediate 5-9  | Dense >=10
C) Alternative: Sparse 2-5 | Intermediate 6-11 | Dense >=12

For each definition:
- paired DeepSets vs SetTransformer comparison
- Delta MAE = MAE_ST - MAE_DS
- paired bootstrap 95% CI
- relative ST gain
- fold-by-fold Delta MAE

Interpretation:
Delta > 0  -> DeepSets better
Delta ~ 0  -> equivalent
Delta < 0  -> SetTransformer better
"""

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

N_BOOT = 5000
SEED = 42

REGIME_SCHEMES = {
    "Main_2-5_6-10_11+": {
        "Sparse":       lambda n: 2 <= n <= 5,
        "Intermediate": lambda n: 6 <= n <= 10,
        "Dense":        lambda n: n >= 11,
    },

    "Alt_2-4_5-9_10+": {
        "Sparse":       lambda n: 2 <= n <= 4,
        "Intermediate": lambda n: 5 <= n <= 9,
        "Dense":        lambda n: n >= 10,
    },

    "Alt_2-5_6-11_12+": {
        "Sparse":       lambda n: 2 <= n <= 5,
        "Intermediate": lambda n: 6 <= n <= 11,
        "Dense":        lambda n: n >= 12,
    }
}

REGIME_ORDER = [
    "Sparse",
    "Intermediate",
    "Dense"
]

# ============================================================
# 1. PREPARE PAIRED EVENT-LEVEL DATA
# ============================================================

ds = pred[
    pred["Model"] == "DeepSets"
][
    [
        "event_id",
        "fold",
        "n_stations",
        "y_true",
        "y_pred"
    ]
].copy()

st = pred[
    pred["Model"] == "SetTransformer"
][
    [
        "event_id",
        "fold",
        "y_pred"
    ]
].copy()

ds = ds.rename(
    columns={
        "y_pred": "y_pred_DS"
    }
)

st = st.rename(
    columns={
        "y_pred": "y_pred_ST"
    }
)

# Include fold in the merge as an additional consistency check
paired_rob = ds.merge(
    st,
    on=["event_id", "fold"],
    how="inner",
    validate="one_to_one"
)

paired_rob["AE_DS"] = np.abs(
    paired_rob["y_true"]
    - paired_rob["y_pred_DS"]
)

paired_rob["AE_ST"] = np.abs(
    paired_rob["y_true"]
    - paired_rob["y_pred_ST"]
)

paired_rob["Delta"] = (
    paired_rob["AE_ST"]
    - paired_rob["AE_DS"]
)

print("=" * 100)
print("ROBUSTNESS OF COVERAGE-REGIME BOUNDARIES")
print("=" * 100)

print(
    f"\nPaired events: {len(paired_rob):,}"
)

print(
    f"Unique events: "
    f"{paired_rob['event_id'].nunique():,}"
)

print(
    f"Folds: "
    f"{sorted(paired_rob['fold'].unique())}"
)

# ============================================================
# 2. REGIME ASSIGNMENT
# ============================================================

def assign_regime(n, scheme):

    for regime in REGIME_ORDER:

        if scheme[regime](n):
            return regime

    return np.nan


# ============================================================
# 3. PAIRED BOOTSTRAP FUNCTION
# ============================================================

def paired_bootstrap_delta(
    delta,
    n_boot=5000,
    seed=42
):

    delta = np.asarray(delta)

    delta = delta[
        np.isfinite(delta)
    ]

    n = len(delta)

    if n == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    rng = np.random.default_rng(seed)

    observed = delta.mean()

    boot = np.empty(n_boot)

    for b in range(n_boot):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        boot[b] = delta[idx].mean()

    ci_low, ci_high = np.percentile(
        boot,
        [2.5, 97.5]
    )

    p_st_better = np.mean(
        boot < 0
    )

    return (
        observed,
        ci_low,
        ci_high,
        p_st_better
    )


# ============================================================
# 4. OVERALL ROBUSTNESS ACROSS CUTOFF DEFINITIONS
# ============================================================

overall_rows = []

for scheme_idx, (scheme_name, scheme) in enumerate(
    REGIME_SCHEMES.items()
):

    dscheme = paired_rob.copy()

    dscheme["Regime"] = (
        dscheme["n_stations"]
        .apply(
            lambda n:
            assign_regime(n, scheme)
        )
    )

    for regime_idx, regime in enumerate(
        REGIME_ORDER
    ):

        d = dscheme[
            dscheme["Regime"] == regime
        ].copy()

        n = len(d)

        mae_ds = d["AE_DS"].mean()
        mae_st = d["AE_ST"].mean()

        (
            delta,
            ci_low,
            ci_high,
            p_st_better
        ) = paired_bootstrap_delta(
            d["Delta"].values,
            n_boot=N_BOOT,
            seed=(
                SEED
                + 100 * scheme_idx
                + regime_idx
            )
        )

        relative_gain = (
            (mae_ds - mae_st)
            / mae_ds
            * 100
        )

        # Direction classification based on CI
        if ci_high < 0:
            conclusion = "ST better"
        elif ci_low > 0:
            conclusion = "DS better"
        else:
            conclusion = "Equivalent"

        overall_rows.append({
            "Scheme": scheme_name,
            "Regime": regime,
            "N": n,
            "MAE_DS": mae_ds,
            "MAE_ST": mae_st,
            "Delta_MAE_ST_minus_DS": delta,
            "Relative_gain_ST_pct": relative_gain,
            "CI95_low": ci_low,
            "CI95_high": ci_high,
            "P_boot_ST_better": p_st_better,
            "Conclusion": conclusion
        })


robust_overall = pd.DataFrame(
    overall_rows
)

robust_overall["Regime"] = pd.Categorical(
    robust_overall["Regime"],
    categories=REGIME_ORDER,
    ordered=True
)

print("\n" + "=" * 100)
print("OVERALL — ALTERNATIVE COVERAGE DEFINITIONS")
print("=" * 100)

print(
    robust_overall.to_string(
        index=False,
        formatters={
            "MAE_DS":
                "{:.5f}".format,

            "MAE_ST":
                "{:.5f}".format,

            "Delta_MAE_ST_minus_DS":
                "{:+.5f}".format,

            "Relative_gain_ST_pct":
                "{:+.2f}".format,

            "CI95_low":
                "{:+.5f}".format,

            "CI95_high":
                "{:+.5f}".format,

            "P_boot_ST_better":
                "{:.4f}".format
        }
    )
)


# ============================================================
# 5. FOLD-BY-FOLD ANALYSIS
#    Use MAIN regime definition
# ============================================================

main_name = "Main_2-5_6-10_11+"
main_scheme = REGIME_SCHEMES[
    main_name
]

dmain = paired_rob.copy()

dmain["Regime"] = (
    dmain["n_stations"]
    .apply(
        lambda n:
        assign_regime(
            n,
            main_scheme
        )
    )
)

fold_rows = []

for fold in sorted(
    dmain["fold"].unique()
):

    dfold = dmain[
        dmain["fold"] == fold
    ]

    for regime in REGIME_ORDER:

        d = dfold[
            dfold["Regime"] == regime
        ]

        if len(d) == 0:
            continue

        mae_ds = d["AE_DS"].mean()
        mae_st = d["AE_ST"].mean()

        delta = (
            mae_st
            - mae_ds
        )

        relative_gain = (
            (mae_ds - mae_st)
            / mae_ds
            * 100
        )

        fold_rows.append({
            "Fold": fold,
            "Regime": regime,
            "N": len(d),
            "MAE_DS": mae_ds,
            "MAE_ST": mae_st,
            "Delta_MAE_ST_minus_DS": delta,
            "Relative_gain_ST_pct": relative_gain
        })


robust_folds = pd.DataFrame(
    fold_rows
)

robust_folds["Regime"] = pd.Categorical(
    robust_folds["Regime"],
    categories=REGIME_ORDER,
    ordered=True
)

robust_folds = robust_folds.sort_values(
    ["Regime", "Fold"]
)

print("\n" + "=" * 100)
print("MAIN DEFINITION — FOLD-BY-FOLD")
print("=" * 100)

print(
    robust_folds.to_string(
        index=False,
        formatters={
            "MAE_DS":
                "{:.5f}".format,

            "MAE_ST":
                "{:.5f}".format,

            "Delta_MAE_ST_minus_DS":
                "{:+.5f}".format,

            "Relative_gain_ST_pct":
                "{:+.2f}".format
        }
    )
)


# ============================================================
# 6. DIRECTIONAL CONSISTENCY ACROSS FOLDS
# ============================================================

consistency_rows = []

for regime in REGIME_ORDER:

    d = robust_folds[
        robust_folds["Regime"] == regime
    ]

    deltas = (
        d["Delta_MAE_ST_minus_DS"]
        .to_numpy()
    )

    n_folds = len(d)

    ds_better = np.sum(
        deltas > 0
    )

    st_better = np.sum(
        deltas < 0
    )

    consistency_rows.append({
        "Regime": regime,
        "N_folds": n_folds,
        "Folds_DS_better": ds_better,
        "Folds_ST_better": st_better,
        "Mean_Delta": deltas.mean(),
        "Min_Delta": deltas.min(),
        "Max_Delta": deltas.max()
    })


fold_consistency = pd.DataFrame(
    consistency_rows
)

print("\n" + "=" * 100)
print("DIRECTIONAL CONSISTENCY ACROSS FOLDS")
print("=" * 100)

print(
    fold_consistency.to_string(
        index=False,
        formatters={
            "Mean_Delta":
                "{:+.5f}".format,

            "Min_Delta":
                "{:+.5f}".format,

            "Max_Delta":
                "{:+.5f}".format
        }
    )
)


# ============================================================
# 7. SAVE
# ============================================================

path_overall = (
    OUT_DIR
    / "coverage_boundary_robustness.csv"
)

path_folds = (
    OUT_DIR
    / "coverage_regime_fold_robustness.csv"
)

path_consistency = (
    OUT_DIR
    / "coverage_regime_fold_consistency.csv"
)

robust_overall.to_csv(
    path_overall,
    index=False
)

robust_folds.to_csv(
    path_folds,
    index=False
)

fold_consistency.to_csv(
    path_consistency,
    index=False
)

print("\n" + "=" * 100)
print("SAVED")
print("=" * 100)

print(path_overall)
print(path_folds)
print(path_consistency)

FINAL FIGURES — Coverage-dependent set-model performance

In [ ]:
# %% ============================================================
# FINAL FIGURES — Coverage-dependent set-model performance
# Fig. 2: MAE by coverage regime
# Fig. 3: Set Transformer vs Deep Sets — ΔMAE + 95% bootstrap CI
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
RESULTS_DIR = OUT_DIR

REGIME_FILE = RESULTS_DIR / "coverage_regime_metrics.csv"
BOOT_FILE   = RESULTS_DIR / "bootstrap_ST_vs_DS_fine_coverage.csv"

FIG_DIR = RESULTS_DIR / "final_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Load final results
# ------------------------------------------------------------
regime = pd.read_csv(REGIME_FILE)
boot   = pd.read_csv(BOOT_FILE)

print("coverage_regime_metrics.csv")
print(regime.to_string(index=False))

print("\nbootstrap_ST_vs_DS_fine_coverage.csv")
print(boot.to_string(index=False))

print("\nColumns:")
print("Regime:", regime.columns.tolist())
print("Bootstrap:", boot.columns.tolist())


# ================================================================
# Helper: robust column lookup
# ================================================================

def find_col(df, candidates):
    """Return first matching column name, case-insensitive."""
    lookup = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    raise KeyError(
        f"None of {candidates} found.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ================================================================
# FIGURE 2 — MAE BY COVERAGE REGIME
# ================================================================

regime_col = find_col(
    regime,
    ["regime", "coverage_regime", "coverage"]
)

model_col = find_col(
    regime,
    ["model", "Model"]
)

mae_col = find_col(
    regime,
    ["MAE", "mae"]
)

n_col = find_col(
    regime,
    ["N", "n", "n_events"]
)

# ------------------------------------------------------------
# Normalize regime labels only for plotting
# ------------------------------------------------------------

def normalize_regime(x):
    s = str(x).strip().lower()

    if "sparse" in s or "2-5" in s or "2–5" in s:
        return "Sparse\n(2–5)"

    if "intermediate" in s or "6-10" in s or "6–10" in s:
        return "Intermediate\n(6–10)"

    if (
        "dense" in s
        or ">=11" in s
        or "≥11" in s
        or "11+" in s
    ):
        return "Dense\n($\\geq$11)"

    return str(x)


regime = regime.copy()
regime["_plot_regime"] = regime[regime_col].map(normalize_regime)

regime_order = [
    "Sparse\n(2–5)",
    "Intermediate\n(6–10)",
    "Dense\n($\\geq$11)"
]

# Desired conceptual model order
preferred_models = [
    "MeanPool-MLP",
    "DeepSets",
    "SetTransformer"
]

available_models = regime[model_col].astype(str).unique().tolist()

# Allow common spelling variants
model_alias = {}

for m in available_models:
    compact = (
        m.lower()
        .replace(" ", "")
        .replace("-", "")
        .replace("_", "")
    )

    if "meanpool" in compact:
        model_alias[m] = "MeanPool-MLP"
    elif "deepset" in compact:
        model_alias[m] = "DeepSets"
    elif "settransformer" in compact:
        model_alias[m] = "SetTransformer"
    else:
        model_alias[m] = m

regime["_plot_model"] = regime[model_col].map(model_alias)

# Pivot to ensure correct alignment
pivot = regime.pivot_table(
    index="_plot_regime",
    columns="_plot_model",
    values=mae_col,
    aggfunc="first"
)

pivot = pivot.reindex(regime_order)

models = [
    m for m in preferred_models
    if m in pivot.columns
]

# Event counts per regime
counts = (
    regime.groupby("_plot_regime")[n_col]
    .first()
    .reindex(regime_order)
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.4, 4.8))

x = np.arange(len(regime_order))
width = 0.23

for i, model in enumerate(models):

    values = pivot[model].values.astype(float)

    offset = (i - (len(models)-1)/2) * width

    bars = ax.bar(
        x + offset,
        values,
        width,
        label=model
    )

    # Exact MAE above bars
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.0015,
            f"{value:.4f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

ax.set_xticks(x)
ax.set_xticklabels(regime_order)

ax.set_ylabel("MAE (magnitude units)")
ax.set_xlabel("Event-level station coverage")

# Add N below/near category labels using axis coordinates
for xi, reg in enumerate(regime_order):
    if reg in counts.index and pd.notna(counts.loc[reg]):
        ax.text(
            xi,
            -0.25,
            f"$N$ = {int(counts.loc[reg]):,}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=9
        )

ax.legend(
    frameon=False,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.1)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(
    0,
    max(regime[mae_col].astype(float)) * 1.15
)

fig.tight_layout()

fig2_pdf = FIG_DIR / "Fig2_MAE_by_coverage_regime.pdf"
fig2_png = FIG_DIR / "Fig2_MAE_by_coverage_regime.png"

fig.savefig(fig2_pdf, bbox_inches="tight")
fig.savefig(fig2_png, dpi=600, bbox_inches="tight")

plt.show()

print("\nSaved Fig. 2:")
print(fig2_pdf)
print(fig2_png)


# ================================================================
# FIGURE 3 — ΔMAE ST - DS WITH BOOTSTRAP 95% CI
# ================================================================

coverage_col = find_col(
    boot,
    [
        "coverage",
        "coverage_bin",
        "n_stations",
        "station_bin",
        "bin"
    ]
)

delta_col = find_col(
    boot,
    [
        "Delta_MAE_ST_minus_DS",
        "delta_mae",
        "Delta_MAE",
        "delta"
    ]
)

low_col = find_col(
    boot,
    [
        "CI_low",
        "ci_low",
        "CI95_low",
        "lower",
        "lower_ci"
    ]
)

high_col = find_col(
    boot,
    [
        "CI_high",
        "ci_high",
        "CI95_high",
        "upper",
        "upper_ci"
    ]
)

b = boot.copy()

# ------------------------------------------------------------
# Preserve the CSV's final scientific binning.
# Expected:
# 2,3,4,5,6,7,8,9,10,11-15,>15
# ------------------------------------------------------------

def coverage_sort_key(v):
    s = str(v).strip()

    if s.startswith(">"):
        return 1000

    if "-" in s or "–" in s:
        s2 = s.replace("–", "-")
        try:
            return float(s2.split("-")[0])
        except:
            return 900

    try:
        return float(s)
    except:
        return 999


b["_sort"] = b[coverage_col].map(coverage_sort_key)
b = b.sort_values("_sort").reset_index(drop=True)

labels = b[coverage_col].astype(str).tolist()

delta = b[delta_col].astype(float).to_numpy()
low   = b[low_col].astype(float).to_numpy()
high  = b[high_col].astype(float).to_numpy()

# Asymmetric CI distances
yerr = np.vstack([
    delta - low,
    high - delta
])

x = np.arange(len(b))

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.6, 4.8))

ax.errorbar(
    x,
    delta,
    yerr=yerr,
    fmt="o-",
    linewidth=1.6,
    markersize=5,
    capsize=4
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.2
)

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_xlabel("Number of active stations")
ax.set_ylabel(
    r"$\Delta$MAE = MAE$_{\mathrm{ST}}$ $-$ MAE$_{\mathrm{DS}}$"
)

# Explanatory labels — sign interpretation
ylim = ax.get_ylim()
yrange = ylim[1] - ylim[0]

ax.text(
    0.07,
    0.95,
    "Lower MAE: Deep Sets",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9
)

ax.text(
    0.07,
    0.05,
    "Lower MAE: Set Transformer",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=9
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

fig3_pdf = FIG_DIR / "Fig3_ST_vs_DS_bootstrap_coverage.pdf"
fig3_png = FIG_DIR / "Fig3_ST_vs_DS_bootstrap_coverage.png"

fig.savefig(fig3_pdf, bbox_inches="tight")
fig.savefig(fig3_png, dpi=600, bbox_inches="tight")

plt.show()

print("\nSaved Fig. 3:")
print(fig3_pdf)
print(fig3_png)


# ================================================================
# FINAL CHECK
# ================================================================

print("\n" + "="*72)
print("FINAL FIGURES GENERATED")
print("="*72)

print("\nFig. 2 — MAE by final coverage regime")
print("  Sparse:       2–5 stations")
print("  Intermediate: 6–10 stations")
print("  Dense:        >=11 stations")

print("\nFig. 3 — paired bootstrap comparison")
print("  ΔMAE > 0 : Deep Sets lower MAE")
print("  ΔMAE < 0 : Set Transformer lower MAE")
print("  Error bars: 95% bootstrap confidence intervals")

print("\nOutput directory:")
print(FIG_DIR)